# REINFORCE on CartPole with Raw Returns

This notebook introduces **REINFORCE**, a simple policy-gradient algorithm. Instead of learning action-values, the agent learns a policy directly:

$$\pi(a \mid s;\theta).$$

The policy network receives a state and produces a probability for each action. During training, actions are sampled from these probabilities. At the end of each episode, the network is updated so that actions followed by larger returns become more likely.

This is plain episodic REINFORCE: there is no replay buffer, target network, critic, or learned baseline. In this version, discounted returns are used at their original scale without normalization.

In [ ]:
import gymnasium as gym
import numpy as np
import random
import time

import torch
import torch.nn as nn
from matplotlib import pyplot as plt

## Configuration

In [ ]:
# Training configuration
NUM_EPISODES = 500
MAX_STEPS_PER_EPISODE = 500
GAMMA = 0.9
LEARNING_RATE = 0.001
HIDDEN_SIZE = 128

SEED = 42

## Environment

`CartPole-v1` returns four continuous observations:

1. cart position;
2. cart velocity;
3. pole angle;
4. pole angular velocity.

There are two actions: `0` pushes the cart left and `1` pushes it right. The agent receives reward $+1$ at every step. An episode terminates if the pole tilts too far or the cart moves too far from the centre. It is truncated after 500 steps; Gymnasium considers an average return of 475 over 100 episodes to be solved.

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

env = gym.make(
    "CartPole-v1",
    max_episode_steps=MAX_STEPS_PER_EPISODE,
)
env.action_space.seed(SEED)
env.observation_space.seed(SEED)

state_size = env.observation_space.shape[0]
action_size = env.action_space.n

print("State size:", state_size)
print("Action size:", action_size)
print("Maximum episode steps:", env.spec.max_episode_steps)
print("Reward threshold:", env.spec.reward_threshold)
print("Device:", device)

## Policy network

The network receives the four observations and returns two **logits**, one for each action:

$$[x, \dot{x}, \theta, \dot{\theta}]\;\longrightarrow\;[z_{\mathrm{left}}, z_{\mathrm{right}}].$$

A softmax converts the logits into action probabilities. We leave the softmax outside the network because PyTorch's categorical distribution can perform this conversion safely.

In [ ]:
class PolicyNetwork(nn.Module):
    """Map a CartPole observation to one logit per action."""

    def __init__(self, state_size, action_size, hidden_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, action_size),
        )

    def forward(self, state):
        return self.net(state)

In [ ]:
policy_net = PolicyNetwork(state_size, action_size, HIDDEN_SIZE).to(device)
optimizer = torch.optim.Adam(policy_net.parameters(), lr=LEARNING_RATE)

## Policy helpers

During training, the agent samples from the policy. Sampling supplies exploration automatically, so REINFORCE does not need an $\varepsilon$-greedy policy. The log-probability of each sampled action is kept for the policy-gradient update.

During evaluation, the greedy action is simply the action with the largest logit.

In [ ]:
def sample_action(state):
    """Sample an action and return its log-probability."""
    state_tensor = torch.as_tensor(
        state, dtype=torch.float32, device=device
    ).unsqueeze(0)

    logits = policy_net(state_tensor)
    distribution = torch.distributions.Categorical(logits=logits)
    action_tensor = distribution.sample()
    log_probability = distribution.log_prob(action_tensor).squeeze()

    action = int(action_tensor.item())
    return action, log_probability


def greedy_action(state):
    """Choose the action with the largest policy logit."""
    state_tensor = torch.as_tensor(
        state, dtype=torch.float32, device=device
    ).unsqueeze(0)

    with torch.no_grad():
        logits = policy_net(state_tensor)

    return int(logits.argmax(dim=1).item())

### Check the policy output

A minibatch is not needed for REINFORCE, but a small batch is useful for checking the network. For each observation, the policy returns two logits and therefore two probabilities that sum to one.

In [ ]:
example_states = torch.zeros((4, state_size), device=device)
example_logits = policy_net(example_states)
example_probabilities = torch.softmax(example_logits, dim=1)

print("Observations:       ", tuple(example_states.shape))
print("Policy logits:      ", tuple(example_logits.shape))
print("Action probabilities:", tuple(example_probabilities.shape))
print("Probability sums:   ", example_probabilities.sum(dim=1))

## Discounted returns

After an episode, the return from time $t$ is calculated by working backwards through its rewards:

$$G_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \cdots.$$

The discounted returns are used directly, without centering or scaling them.

In [ ]:
def calculate_returns(rewards):
    """Calculate the raw discounted reward-to-go."""
    discounted_returns = []
    running_return = 0.0

    # Work backwards, then insert each return at the front.
    for reward in reversed(rewards):
        running_return = reward + GAMMA * running_return
        discounted_returns.insert(0, running_return)

    returns_tensor = torch.as_tensor(
        discounted_returns, dtype=torch.float32, device=device
    )

    return returns_tensor

In [ ]:
example_rewards = [1.0, 1.0, 1.0, 1.0]
example_returns = calculate_returns(example_rewards)

print("Number of rewards:", len(example_rewards))
print("Number of returns:", len(example_returns))
print("Raw discounted returns:", example_returns)
print("All returns finite:", bool(torch.isfinite(example_returns).all()))

## Training

For each episode, we sample actions and store their log-probabilities and rewards. Once the episode finishes, REINFORCE minimizes

$$L(\theta) = -\sum_t G_t \log \pi(A_t \mid S_t;\theta).$$

The minus sign lets the optimizer perform gradient descent while increasing the probability of actions associated with larger returns. There is one network update per complete episode.

In [ ]:
rewards_all_episodes = []
steps_all_episodes = []
losses = []

start_time = time.time()

for episode in range(NUM_EPISODES):
    if episode == 0:
        state, info = env.reset(seed=SEED)
    else:
        state, info = env.reset()

    episode_rewards = []
    log_probabilities = []

    for step in range(MAX_STEPS_PER_EPISODE):
        action, log_probability = sample_action(state)
        next_state, reward, terminated, truncated, info = env.step(action)

        log_probabilities.append(log_probability)
        episode_rewards.append(reward)
        state = next_state

        if terminated or truncated:
            break

    # Calculate one return for each action taken in the episode.
    discounted_returns = calculate_returns(episode_rewards)

    # Actions with larger returns should become more likely.
    policy_loss = -(
        torch.stack(log_probabilities) * discounted_returns
    ).sum()

    optimizer.zero_grad()
    policy_loss.backward()
    optimizer.step()

    episode_reward = sum(episode_rewards)
    steps_taken = len(episode_rewards)
    rewards_all_episodes.append(episode_reward)
    steps_all_episodes.append(steps_taken)
    losses.append(policy_loss.item())

    if episode == 0 or (episode + 1) % 50 == 0:
        recent_mean = np.mean(rewards_all_episodes[-20:])
        print(
            f"Episode {episode + 1:4d}: "
            f"reward={episode_reward:5.0f}, "
            f"mean20={recent_mean:6.1f}"
        )

training_minutes = (time.time() - start_time) / 60
print(f"Training time: {training_minutes:.2f} minutes")
print(f"Policy updates: {len(losses)}")
print("All losses finite:", bool(np.isfinite(losses).all()))

## Learning curves

CartPole's reward equals the number of steps survived, so the return and episode length contain the same values. They are shown separately to match the earlier notebooks and to emphasize their different meanings.

In [ ]:
window = 20
rolling_rewards = np.convolve(
    np.array(rewards_all_episodes),
    np.ones(window) / window,
    mode="valid",
)

plt.figure(figsize=(8, 5))
plt.plot(np.arange(window, NUM_EPISODES + 1), rolling_rewards)
plt.axhline(475, color="tab:red", linestyle="--", label="Solved threshold")
plt.title(f"Rolling Average Return (window = {window} episodes)")
plt.xlabel("Episode")
plt.ylabel("Average return")
plt.ylim(0, MAX_STEPS_PER_EPISODE + 10)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(steps_all_episodes, alpha=0.65)
plt.title("Number of Steps per Episode")
plt.xlabel("Episode")
plt.ylabel("Steps")
plt.ylim(0, MAX_STEPS_PER_EPISODE + 10)
plt.grid(True)
plt.tight_layout()
plt.show()

## Evaluate the greedy policy

Training samples actions from the policy, so its returns do not directly measure the final greedy policy. We now choose the action with the largest logit and evaluate it over 100 fresh episodes.

In [ ]:
policy_net.eval()
evaluation_returns = []

for evaluation_episode in range(100):
    state, info = env.reset(seed=SEED + 1000 + evaluation_episode)
    episode_return = 0.0

    for step in range(MAX_STEPS_PER_EPISODE):
        action = greedy_action(state)
        state, reward, terminated, truncated, info = env.step(action)
        episode_return += reward

        if terminated or truncated:
            break

    evaluation_returns.append(episode_return)

evaluation_returns = np.array(evaluation_returns)
mean_return = evaluation_returns.mean()

print(f"Mean return:     {mean_return:.2f}")
print(f"Standard dev.:   {evaluation_returns.std():.2f}")
print(f"Minimum return:  {evaluation_returns.min():.0f}")
print(f"Maximum return:  {evaluation_returns.max():.0f}")
print(f"Solved (>= 475): {mean_return >= 475}")

## One greedy rollout

Instead of opening a separate rendering window, record one greedy episode and plot two important state variables. A successful policy keeps both the cart position and pole angle close to zero.

In [ ]:
state, info = env.reset(seed=SEED + 2000)
rollout_states = [state.copy()]
rollout_return = 0.0

for step in range(MAX_STEPS_PER_EPISODE):
    action = greedy_action(state)
    state, reward, terminated, truncated, info = env.step(action)
    rollout_states.append(state.copy())
    rollout_return += reward

    if terminated or truncated:
        break

rollout_states = np.array(rollout_states)
time_steps = np.arange(len(rollout_states))

fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
axes[0].plot(time_steps, rollout_states[:, 0])
axes[0].axhline(-2.4, color="red", linestyle="--", label="Position limits")
axes[0].axhline(2.4, color="red", linestyle="--")
axes[0].set_ylabel("Cart position")
axes[0].grid(True)
axes[0].legend()

axes[1].plot(time_steps, np.degrees(rollout_states[:, 2]))
axes[1].axhline(-12.0, color="red", linestyle="--", label="Pole angle limits")
axes[1].axhline(12.0, color="red", linestyle="--")
axes[1].set_xlabel("Step")
axes[1].set_ylabel("Pole angle (degrees)")
axes[1].grid(True)
axes[1].legend()

fig.suptitle(f"Greedy Rollout (return = {rollout_return:.0f})")
fig.tight_layout()
plt.show()

print(f"Rollout return: {rollout_return:.0f}")
print(f"Rollout steps:  {len(rollout_states) - 1}")

## Save the trained network

The learned policy is stored as the PyTorch network's state dictionary. The same `PolicyNetwork` architecture must be created before these weights are loaded later. We also check that the reloaded network chooses the same greedy action as the trained network.

In [ ]:
MODEL_PATH = "reinforce_cartpole_raw_returns_policy.pth"
torch.save(policy_net.state_dict(), MODEL_PATH)

# Check that the saved state dictionary can be loaded.
saved_weights = torch.load(MODEL_PATH, map_location=device, weights_only=True)
check_net = PolicyNetwork(state_size, action_size, HIDDEN_SIZE).to(device)
check_net.load_state_dict(saved_weights)
check_net.eval()

check_state, info = env.reset(seed=SEED + 3000)
check_state_tensor = torch.as_tensor(
    check_state, dtype=torch.float32, device=device
).unsqueeze(0)

with torch.no_grad():
    original_action = policy_net(check_state_tensor).argmax(dim=1).item()
    reloaded_action = check_net(check_state_tensor).argmax(dim=1).item()

print("Saved and reloaded:", MODEL_PATH)
print("Greedy actions match:", original_action == reloaded_action)
env.close()